# EDA with Pandas
Exploratory Data Analysis -- understand your data before modelling.

In [ ]:
import micropip
await micropip.install(['pandas','matplotlib','numpy','seaborn'])
print('Ready!')

## The EDA Checklist

1. **Shape** -- rows, columns
2. **Dtypes** -- numeric, categorical, datetime
3. **Missing values** -- count, pattern
4. **Distributions** -- hist, box, violin
5. **Correlations** -- heatmap
6. **Outliers** -- Z-score, IQR
7. **Target variable** -- class balance, distribution

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age':        np.random.normal(35, 10, n).clip(18, 80).astype(int),
    'salary':     np.random.exponential(50000, n).clip(20000, 200000),
    'experience': np.random.randint(0, 30, n),
    'department': np.random.choice(['Eng','Sales','HR','Finance','Product'], n),
    'rating':     np.random.uniform(2, 5, n).round(1),
    'promoted':   np.random.choice([0, 1], n, p=[0.75, 0.25]),
})
df.loc[np.random.choice(n, 20), 'salary'] = np.nan
print("Shape:", df.shape)
print("Dtypes:"); print(df.dtypes)
print("\nFirst 3 rows:"); print(df.head(3))

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nSummary statistics:")
print(df.describe().round(0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].hist(df['salary'].dropna(), bins=30, color='#6b21a8', alpha=0.7, edgecolor='white')
axes[0,0].set_title('Salary Distribution')
dept_groups = [df[df['department']==d]['age'].dropna() for d in df['department'].unique()]
dept_labels = list(df['department'].unique())
axes[0,1].boxplot(dept_groups, labels=dept_labels, vert=False)
axes[0,1].set_title('Age by Department')
promo_rate = df.groupby('department')['promoted'].mean().sort_values()
axes[1,0].barh(promo_rate.index, promo_rate.values, color='#059669')
axes[1,0].set_title('Promotion Rate by Department')
axes[1,1].scatter(df['experience'], df['salary'], c=df['promoted'], cmap='RdYlGn', alpha=0.5, s=20)
axes[1,1].set_title('Salary vs Experience')
axes[1,1].set_xlabel('Experience'); axes[1,1].set_ylabel('Salary')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()
plt.figure(figsize=(7, 5))
im = plt.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
cols = list(corr.columns)
plt.xticks(range(len(cols)), cols, rotation=45, ha='right')
plt.yticks(range(len(cols)), cols)
for i in range(len(corr)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center', fontsize=9)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Outlier detection -- IQR method
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df['salary'] < lower) | (df['salary'] > upper)]
print(f"IQR range: [{lower:.0f}, {upper:.0f}]")
print(f"IQR outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
z = (df['salary'] - df['salary'].mean()) / df['salary'].std()
print(f"Z-score outliers (|z|>3): {(z.abs()>3).sum()}")

---
**Your turn:** Add a `salary_bucket` column using `pd.cut()` with 4 bins and show promotion rate per bucket.